In [6]:
import ROOT as rt
import ctypes
%jsroot on
file = rt.TFile("e21062_44S.root")
dtimeN = file.Get("dtimeN")

In [4]:
#44SGated
def decay_parent(x, par):
    """
    a = amplitude
    b = decay constant (lambda)
    bkgrd = constant background
    """
    a = par[0]
    b = par[1]
    bkgrd = par[2]
    t = x[0]

    return a * rt.TMath.Exp(-b * t) + bkgrd

def Decayp(hist):
    
    MaxX = hist.GetXaxis().GetXmax();
    #MinX = hist.GetXaxis().GetXmin();
    # Axis range
    MinX = 1
    #MaxX = 
    
    print("Min X: ", MinX)
    print("Max X: ", MaxX)
    
    # Canvas
    fitgraph = rt.TCanvas("fits", "fits", 800, 600)
    fitgraph.Clear()

    # Parent half-life guess
    Parent_halflife = rt.TMath.Log(2) / 101.0

    # Define TF1
    decayp = rt.TF1("decayp", decay_parent, MinX, MaxX, 3)

    decayp.SetParName(0, "source #")
    decayp.SetParName(1, "Source lambda")
    decayp.SetParName(2, "shift")

    decayp.SetParameters(200, Parent_halflife, 200) # intial conditions

    decayp.SetParLimits(0, 0, 500) # source
    decayp.SetParLimits(1, 0, 1) # lambda
    decayp.SetParLimits(2, 0, 400) # bckgrnd

    # Print initial parameters
    for i in range(decayp.GetNpar()):
        print(f"Initial p{i} = {decayp.GetParameter(i)}")

    # Perform fit
    hist.Fit("decayp", "RS", "", MinX, MaxX)
    #hist.Print("V")
    
    # Print parameter limits
    for i in range(decayp.GetNpar()):
        par_min = ctypes.c_double()
        par_max = ctypes.c_double()
        decayp.GetParLimits(i, par_min, par_max)
        print(f"Parameter {i} ({decayp.GetParName(i)}) limits: [{par_min}, {par_max}]")

    # Half-life calculation
    lambda_fit = decayp.GetParameter(1)
    lambda_err = decayp.GetParError(1)

    decayhl = rt.TMath.Log(2) / lambda_fit
    decay_error = (rt.TMath.Log(2) / (lambda_fit**2)) * lambda_err

    print(f"Decay: {decayhl:.6f} ms ± {decay_error:.6f} ms\n")

    # Chi2 / NDF
    chi2 = decayp.GetChisquare()
    ndf = decayp.GetNDF()
    print(f"Chi^2/NDF: {chi2/ndf:.6f}\n")

    # Constant background function
    bckgrnd = rt.TF1("bckgrnd", "[0]", MinX, MaxX)
    bckgrnd.SetParameter(0, decayp.GetParameter(2))
    bckgrnd.SetLineColor(rt.kBlack)

    # Legend
    legend = rt.TLegend(0.7, 0.4, 0.9, 0.7)
    legend.AddEntry(hist, "Data", "E")
    legend.AddEntry(decayp, "Fit", "l")
    legend.AddEntry(bckgrnd, "backgrnd", "l")

    # Styling
    hist.SetTitle("Decay of 44S Gated on 2789 keV;Time (ms);Counts")
    hist.GetXaxis().CenterTitle(True)
    hist.GetYaxis().CenterTitle(True)
    hist.SetLineColor(rt.kBlue)

    # Draw
    hist.Draw("E")
    decayp.SetLineColor(rt.kRed)
    decayp.Draw("same")
    bckgrnd.Draw("same")
    fitgraph.Draw()
    hist.Print("V")
    # legend.Draw()  # Uncomment if desired

    fitgraph.SaveAs("GammaNTest.png")

In [7]:
gamma_canvas = rt.TCanvas("gamma_canvas", "gamma_2789")
gamma_2789 = dtimeN.ProjectionX("Decay_Curve_2789keV",2783,2789)
#Decayp(gamma_2789)
#gamma_canvas.Update()
#gamma_canvas.Draw()

bckgrnd_2789 = dtimeN.ProjectionX("Decay_Curve_2789keV_bckgnd",2811,2818)
gamma_2789.Add(bckgrnd_2789,-1)
gamma_2789.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: gamma_canvas


In [8]:
Decayp(gamma_2789)
gamma_canvas.Update()
gamma_canvas.Draw()

Min X:  1
Max X:  1000.0
Initial p0 = 200.0
Initial p1 = 0.0068628433718806465
Initial p2 = 200.0
Parameter 0 (source #) limits: [c_double(0.0), c_double(500.0)]
Parameter 1 (Source lambda) limits: [c_double(0.0), c_double(1.0)]
Parameter 2 (shift) limits: [c_double(0.0), c_double(400.0)]
Decay: 108.023819 ms ± 25.480785 ms

Chi^2/NDF: 1.133603

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1063.32
NDf                       =          938
Edm                       =  7.60079e-07
NCalls                    =          168
source #                  =      2.44785   +/-   0.335274     	 (limited)
Source lambda             =   0.00641661   +/-   0.00151356   	 (limited)
shift                     =      2.01352   +/-   0.0816649    	 (limited)
TH1.Print Name  = Decay_Curve_2789keV, Entries= 3226, Total sum= 3226


In file included from input_line_93:1:
In file included from /opt/anaconda3/envs/Sulfur/include/CPyCppyy/API.h:26:
In file included from /opt/anaconda3/envs/Sulfur/include/python3.11/Python.h:92:
/opt/anaconda3/envs/Sulfur/include/python3.11/compile.h:8:9: warning: 'Py_single_input' macro redefined [-Wmacro-redefined]
#define Py_single_input 256
        ^
:49:1: note: previous definition is here
   }
^
Info in <TCanvas::Print>: png file GammaNTest.png has been created
